# 03 - Analysis and figures

Runs **locally on your laptop** (or Kaggle CPU). No GPU, no internet, ~1 minute.
Expects the `runs/<person>/results/` folders in place.

Produces every figure as **PDF** (the guidelines require vector, not PNG) plus PNG previews,
and every number as CSV, into `analysis/figures/` and `analysis/tables/`.

Colour choices follow one rule: exposure buckets are an **ordered** variable, so they get a
single-hue ordinal ramp; model sizes and template sets are **categorical**, so they get distinct
hues; the confidence-accuracy gap is **diverging** around zero. All palettes were validated for
colour-vision deficiency before use.

## 1. Setup

In [ ]:

import os, glob, json, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# ---- paper-figure defaults (ACL: \columnwidth ~3.17in, \textwidth ~6.3in) ----
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "grid.linewidth": 0.4, "lines.linewidth": 1.4,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 150, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
COL, FULL = 3.17, 6.3
SURFACE = "#fcfcfb"
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#8a8984"
# categorical slots 1-3 (validated all-pairs, light)
CAT = ["#2a78d6", "#eb6834", "#1baf7a"]
# 5-step ordinal blue ramp (validated: monotone L, adjacent dL >= 0.06)
ORD5 = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]
# diverging blue<->red with neutral gray midpoint
DIV = LinearSegmentedColormap.from_list("bl_rd",
        ["#0d366b", "#2a78d6", "#86b6ef", "#f0efec", "#f0a19f", "#d03b3b", "#7d1f1f"])
SEQ = LinearSegmentedColormap.from_list("blues", ["#eef4fd", "#86b6ef", "#2a78d6", "#0d366b"])

BUCKETS = ["0","1","2","3-4","5-7","8-13","14-25","26-60","61-200","200+"]
BSHOW   = ["0","2","5-7","26-60","200+"]          # 5 representative buckets for line charts
MAIN, FINAL = "LMEnt-1B-6E", 658032

def style(ax, grid="y"):
    ax.set_facecolor(SURFACE)
    if grid: ax.grid(axis=grid, color="#e3e2de", zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, length=2, width=0.6)
    for s in ax.spines.values(): s.set_color("#c9c8c3")
    return ax

def save(fig, name, outdir):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(outdir, f"{name}.{ext}"), format=ext)
    plt.close(fig)
    print("  figure:", name + ".pdf")

def load_results(root="runs"):
    fs = sorted(glob.glob(os.path.join(root, "*", "results", "res__*.parquet")))
    assert fs, f"no result parquets under {root}/*/results/"
    df = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True)
    num = [c for c in df.columns if df[c].dtype.kind == "f"]
    assert not df[num].isna().any().any(), "NaN in results"
    assert df.duplicated(["model","templates","step","fact_id"]).sum() == 0, "duplicate rows"
    n = df.groupby(["model","templates","step"]).fact_id.nunique().unique()
    assert len(n) == 1, f"inconsistent fact counts per checkpoint: {n}"
    df["bucket"] = pd.Categorical(df.bucket, BUCKETS, ordered=True)
    df["gap_f"] = df.conf_norm - df.acc_norm          # fact-level gap
    return df, fs

def gap_table(d, index="bucket", col="step"):
    g = (d.groupby([index, col], observed=True).conf_norm.mean()
         - d.groupby([index, col], observed=True).acc_norm.mean())
    out = g.unstack(col)
    return out.reindex(BUCKETS) if index == "bucket" else out

def boot_ci(x, n=2000, seed=0, stat=np.mean):
    r = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    if len(x) == 0: return (np.nan, np.nan)
    bs = stat(x[r.integers(0, len(x), (n, len(x)))], axis=1)
    return tuple(np.percentile(bs, [2.5, 97.5]))

def ece(conf, correct, bins=10):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    edges = np.linspace(0, 1, bins+1); tot = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i+1]) if i else (conf <= edges[1])
        if m.sum(): tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return tot


In [ ]:
import os, json
FIGDIR, TABDIR = "analysis/figures", "analysis/tables"
os.makedirs(FIGDIR, exist_ok=True); os.makedirs(TABDIR, exist_ok=True)
T, Q, S = {}, {}, {}

## 2. Load results and verify integrity

In [ ]:
df, files = load_results("runs")
print(f"{len(files)} result files, {len(df):,} rows")
print(df.groupby(["model","templates"]).agg(steps=("step","nunique"),
      lo=("step","min"), hi=("step","max")).to_string())

# every runner must have scored the identical probe set
import glob as _g
hashes = {os.path.basename(m): json.load(open(m))["probe_hash"]
          for m in sorted(_g.glob("runs/*/results/manifest__*.json"))}
print("\nprobe hashes:", set(hashes.values()))
assert len(set(hashes.values())) == 1, f"probe sets differ across runners: {hashes}"
Q["probe_hash"] = list(hashes.values())[0]
Q["n_files"], Q["n_rows"] = len(files), len(df)

## 3. Data-quality checks

Three things that would otherwise become wrong claims in the paper.

In [ ]:

print("=" * 70); print("DATA QUALITY CHECKS"); print("=" * 70)
Q = {}

# Flag 1: is bucket "0" really a zero-knowledge control?
b0 = df[(df.model==MAIN)&(df.templates=="main")&(df.step==FINAL)&(df.bucket=="0")]
Q["bucket0_acc"]          = float(b0.acc_norm.mean())
Q["bucket0_n_subject_med"] = float(b0.n_subject.median())
Q["bucket0_subj_absent"]   = float((b0.n_subject==0).mean())
print(f"[1] bucket '0': accuracy {Q['bucket0_acc']:.3f} vs chance 0.100")
print(f"    median subject_num_chunks = {Q['bucket0_n_subject_med']:.0f}; "
      f"{Q['bucket0_subj_absent']:.1%} have the subject entirely absent")
print("    -> subject-answer co-occurrence is 0, but the SUBJECT is still in the corpus,")
print("       so this is a 'no evidence for this fact' control, not 'entity never seen'.")

# a stricter control: subject absent too
strict = df[(df.model==MAIN)&(df.templates=="main")&(df.step==FINAL)&(df.n_subject==0)]
Q["strict_control_n"]   = int(strict.fact_id.nunique())
Q["strict_control_acc"] = float(strict.acc_norm.mean())
Q["strict_control_gap"] = float(strict.conf_norm.mean()-strict.acc_norm.mean())
print(f"    strict control (n_subject==0, n={Q['strict_control_n']}): "
      f"acc {Q['strict_control_acc']:.3f}, gap {Q['strict_control_gap']:+.3f}")

# Flag 2: does the 200+ vs 61-200 inversion replicate across models?
print("\n[2] gap inversion at the top two buckets, each model's final checkpoint:")
inv = {}
for m, st in df[df.templates=="main"].groupby("model").step.max().items():
    d = df[(df.model==m)&(df.templates=="main")&(df.step==st)]
    g = (d.groupby("bucket", observed=True).conf_norm.mean()
         - d.groupby("bucket", observed=True).acc_norm.mean())
    inv[m] = (float(g.get("61-200", np.nan)), float(g.get("200+", np.nan)))
    print(f"    {m:<14} 61-200 {inv[m][0]:+.3f}   200+ {inv[m][1]:+.3f}   "
          f"{'INVERTED' if inv[m][1] > inv[m][0] else 'ordered'}")
Q["inversion"] = inv
print("    -> replicates in all models, so treat as real and explain, not smooth over.")

# Flag 3: the epoch control is LR-schedule confounded at matched steps
print("\n[3] epoch control: 1E finishes its LR decay at 109,672; 6E is mid-schedule there.")
for st in [10000, 100000]:
    a = df[(df.model=="LMEnt-1B-1E")&(df.step==st)]
    b = df[(df.model==MAIN)&(df.templates=="main")&(df.step==st)]
    if len(a) and len(b):
        print(f"    step {st:>6}: 1E acc {a.acc_norm.mean():.3f} | 6E acc {b.acc_norm.mean():.3f}"
              f"   (CONFOUNDED - report final-vs-final instead)")
Q["epoch_note"] = "matched-step comparison confounded by LR schedule; use final-vs-final"


## 4. Headline: the gap scales with exposure

Diverging bars because the quantity crosses zero. The dashed line is the same measurement at
random initialisation - it is flat, which is what makes the trained curve meaningful.

In [ ]:

# ---- Figure 1: headline. Gap by exposure, trained vs random init ----
d  = df[(df.model==MAIN)&(df.templates=="main")]
gf = gap_table(d)[FINAL]; g0 = gap_table(d)[0]
ci = [boot_ci(d[(d.step==FINAL)&(d.bucket==b)].gap_f) for b in BUCKETS]
err = np.array([[gf[b]-lo, hi-gf[b]] for b,(lo,hi) in zip(BUCKETS, ci)]).T

fig, ax = plt.subplots(figsize=(COL, 2.3)); style(ax)
x = np.arange(len(BUCKETS))
cols = [DIV(0.5 + 0.5*np.clip(v/0.22, -1, 1)) for v in gf]
ax.bar(x, gf, color=cols, width=0.72, zorder=3,
       edgecolor=SURFACE, linewidth=0.8)
ax.errorbar(x, gf, yerr=err, fmt="none", ecolor=INK2, elinewidth=0.7, capsize=1.6, zorder=4)
ax.plot(x, g0, ls="--", color=MUTED, lw=1.0, marker="o", ms=2.5, zorder=5,
        label="at initialisation (step 0)")
ax.axhline(0, color=INK, lw=0.7, zorder=2)
ax.set_xticks(x); ax.set_xticklabels(BUCKETS, rotation=45, ha="right")
ax.set_xlabel("subject–answer co-occurrences in pretraining")
ax.set_ylabel("confidence − accuracy")
ax.legend(frameon=False, loc="upper right")
ax.annotate("overconfident", (0.15, gf.iloc[0]+0.035), color=INK2, fontsize=6.5)
ax.annotate("underconfident", (7.0, gf.iloc[-2]-0.055), color=INK2, fontsize=6.5)
save(fig, "fig1_gap_by_exposure", FIGDIR)
T["fig1"] = pd.DataFrame({"bucket":BUCKETS, "gap_final":gf.values, "gap_step0":g0.values,
                          "ci_lo":[c[0] for c in ci], "ci_hi":[c[1] for c in ci]})


## 5. The whole run at once

In [ ]:

# ---- Figure 2: gap over the whole run, all 10 buckets (heatmap) ----
d = df[(df.model==MAIN)&(df.templates=="main")]
G = gap_table(d)
fig, ax = plt.subplots(figsize=(FULL, 2.5))
vmax = float(np.nanmax(np.abs(G.values)))
im = ax.imshow(G.values, aspect="auto", cmap=DIV,
               norm=TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax))
ax.set_yticks(range(len(BUCKETS))); ax.set_yticklabels(BUCKETS)
cols = list(G.columns)
tick = [i for i,c in enumerate(cols) if c in (0,1000,5000,10000,30000,80000,170000,350000,658032)]
ax.set_xticks(tick); ax.set_xticklabels([f"{cols[i]:,}" for i in tick], rotation=45, ha="right")
ax.set_xlabel("training step"); ax.set_ylabel("co-occurrences")
ax.tick_params(length=2, colors=INK2)
cb = fig.colorbar(im, ax=ax, pad=0.012, fraction=0.03)
cb.set_label("confidence − accuracy", fontsize=7); cb.ax.tick_params(labelsize=6, length=2)
cb.outline.set_linewidth(0.4)
save(fig, "fig2_gap_heatmap", FIGDIR)
T["fig2"] = G


## 6. The dissociation

Five representative buckets (ten single-hue lines are not distinguishable - the ramp fails its
adjacent-lightness check at ten steps).

In [ ]:

# ---- Figure 3: the dissociation ----
d = df[(df.model==MAIN)&(df.templates=="main")]
fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.2))
for ax, met, lab in zip(axes, ["acc_norm","conf_norm"], ["accuracy","confidence"]):
    style(ax, grid="both")
    for c, b in zip(ORD5, BSHOW):
        s = d[d.bucket==b].groupby("step")[met].mean()
        ax.plot(s.index, s.values, color=c, marker="o", ms=2.2, label=b)
        ax.annotate(b, (s.index[-1], s.values[-1]), xytext=(3,-1),
                    textcoords="offset points", fontsize=6, color=c, va="center")
    ax.set_xscale("symlog", linthresh=1000)
    ax.set_xticks([0,1000,10000,100000,658032])
    ax.set_xticklabels(["0","1k","10k","100k","658k"])
    ax.set_xlabel("training step"); ax.set_ylabel(lab); ax.set_ylim(0, 0.9)
axes[0].axhline(0.1, ls=":", color=MUTED, lw=0.8)
axes[0].annotate("chance", (1.2e3, 0.115), fontsize=6, color=MUTED)
axes[0].legend(frameon=False, title="co-occurrences", ncol=2, loc="upper left",
               title_fontsize=6.5)
axes[1].set_title("confidence separates only for the most frequent facts", fontsize=7.5, color=INK2)
axes[0].set_title("accuracy separates across the whole range", fontsize=7.5, color=INK2)
save(fig, "fig3_dissociation", FIGDIR)

# pooled version, single panel
fig, ax = plt.subplots(figsize=(COL, 2.0)); style(ax, grid="both")
for c, met, lab in zip(CAT[:2], ["conf_norm","acc_norm"], ["confidence","accuracy"]):
    s = d.groupby("step")[met].mean()
    ax.plot(s.index, s.values, color=c, marker="o", ms=2.2, label=lab)
    ax.annotate(lab, (s.index[-1], s.values[-1]), xytext=(3,0), textcoords="offset points",
                fontsize=6.5, color=c, va="center")
ax.set_xscale("symlog", linthresh=1000)
ax.set_xticks([0,1000,10000,100000,658032])
ax.set_xticklabels(["0","1k","10k","100k","658k"])
ax.set_xlabel("training step")
ax.set_ylabel("pooled over all facts"); ax.set_ylim(0, 0.55)
save(fig, "fig4_pooled_dissociation", FIGDIR)
T["pooled"] = d.groupby("step")[["acc_norm","conf_norm"]].mean()


## 7. What actually happens early

**The pooled "confidence is flat" statement is an averaging artifact.** Confidence *falls* for
rare facts and *rises* steeply for frequent ones, and the two cancel. The real finding is that at
step 1000 confidence is nearly uniform across exposure while accuracy already is not.

In [ ]:

# ---- Figure: at step 1000 confidence is near-uniform across exposure, accuracy is not ----
d = df[(df.model==MAIN)&(df.templates=="main")]
e = d[d.step==1000].groupby("bucket", observed=True)[["acc_norm","conf_norm"]].mean().reindex(BUCKETS)
f = d[d.step==FINAL].groupby("bucket", observed=True)[["acc_norm","conf_norm"]].mean().reindex(BUCKETS)
fig, axes = plt.subplots(1, 2, figsize=(FULL, 2.0), sharey=True)
x = np.arange(len(BUCKETS))
for ax, tab, ttl in zip(axes, [e, f], ["step 1,000", "step 658,032 (final)"]):
    style(ax, grid="both")
    ax.plot(x, tab.conf_norm.values, color=CAT[0], marker="o", ms=2.8, label="confidence")
    ax.plot(x, tab.acc_norm.values, color=CAT[1], marker="s", ms=2.8, label="accuracy")
    ax.fill_between(x, tab.acc_norm.values, tab.conf_norm.values,
                    color=CAT[0], alpha=0.10, lw=0)
    ax.set_xticks(x); ax.set_xticklabels(BUCKETS, rotation=45, ha="right")
    ax.set_xlabel("co-occurrences"); ax.set_title(ttl, fontsize=7.5, color=INK2)
    ax.set_ylim(0, 0.9)
axes[0].set_ylabel("value"); axes[0].legend(frameon=False, loc="upper left")
save(fig, "fig8_early_uniform_confidence", FIGDIR)

rng_c = float(e.conf_norm.max()-e.conf_norm.min()); rng_a = float(e.acc_norm.max()-e.acc_norm.min())
print(f"at step 1000: confidence spans {rng_c:.3f} across buckets, accuracy spans {rng_a:.3f}")
print(f"  confidence for buckets 0..14-25: {e.conf_norm.iloc[:7].min():.3f}-{e.conf_norm.iloc[:7].max():.3f}")
Q["step1000_conf_range"] = rng_c; Q["step1000_acc_range"] = rng_a

# per-bucket change table - the pooled numbers hide opposite-signed movements
ch = pd.DataFrame({
    "conf_1k": d[d.step==1000].groupby("bucket", observed=True).conf_norm.mean().reindex(BUCKETS),
    "conf_end": f.conf_norm, "acc_1k": e.acc_norm, "acc_end": f.acc_norm})
ch["d_conf"] = ch.conf_end - ch.conf_1k; ch["d_acc"] = ch.acc_end - ch.acc_1k
print(); print(ch.round(3).to_string())
T["change"] = ch


## 8. Controls: scale, repetition, wording

In [ ]:

# ---- Figure 5: scale + controls ----
fig, axes = plt.subplots(1, 3, figsize=(FULL, 2.0))
x = np.arange(len(BUCKETS))

ax = style(axes[0])
for c, m in zip(CAT, ["LMEnt-170M-6E","LMEnt-600M-6E","LMEnt-1B-6E"]):
    st = df[(df.model==m)&(df.templates=="main")].step.max()
    dd = df[(df.model==m)&(df.templates=="main")&(df.step==st)]
    g  = (dd.groupby("bucket", observed=True).conf_norm.mean()
          - dd.groupby("bucket", observed=True).acc_norm.mean()).reindex(BUCKETS)
    ax.plot(x, g.values, color=c, marker="o", ms=2.6, label=m.replace("LMEnt-","").replace("-6E",""))
ax.axhline(0, color=INK, lw=0.7); ax.legend(frameon=False, title="model size", title_fontsize=6.5)
ax.set_title("scale does not fix it", fontsize=7.5, color=INK2)

ax = style(axes[1])
for c, (m, lab) in zip(CAT, [("LMEnt-1B-6E","6 epochs"), ("LMEnt-1B-1E","1 epoch")]):
    st = df[(df.model==m)&(df.templates=="main")].step.max()
    dd = df[(df.model==m)&(df.templates=="main")&(df.step==st)]
    g  = (dd.groupby("bucket", observed=True).conf_norm.mean()
          - dd.groupby("bucket", observed=True).acc_norm.mean()).reindex(BUCKETS)
    ax.plot(x, g.values, color=c, marker="o", ms=2.6, label=lab)
ax.axhline(0, color=INK, lw=0.7); ax.legend(frameon=False, title="1B, final ckpt", title_fontsize=6.5)
ax.set_title("repetition does not fix it", fontsize=7.5, color=INK2)

ax = style(axes[2])
for c, ts in zip(CAT, ["main","alt1","alt2"]):
    dd = df[(df.model==MAIN)&(df.templates==ts)&(df.step==FINAL)]
    g  = (dd.groupby("bucket", observed=True).conf_norm.mean()
          - dd.groupby("bucket", observed=True).acc_norm.mean()).reindex(BUCKETS)
    ax.plot(x, g.values, color=c, marker="o", ms=2.6, label=ts)
ax.axhline(0, color=INK, lw=0.7); ax.legend(frameon=False, title="template set", title_fontsize=6.5)
ax.set_title("wording does not explain it", fontsize=7.5, color=INK2)

for ax in axes:
    ax.set_xticks(x[::2]); ax.set_xticklabels([BUCKETS[i] for i in range(0,len(BUCKETS),2)],
                                              rotation=45, ha="right")
    ax.set_xlabel("co-occurrences")
axes[0].set_ylabel("confidence − accuracy")
save(fig, "fig5_controls", FIGDIR)


## 9. Mechanism and calibration

In [ ]:

# ---- Figure 6: mechanism ----
w = df[(df.model==MAIN)&(df.templates=="main")&(df.acc_norm==0)&(df.step>0)]
cp = w.groupby("bucket", observed=True).chose_corpus_prior.mean().reindex(BUCKETS)
mp = w.groupby("bucket", observed=True).chose_model_prior.mean().reindex(BUCKETS)
fig, ax = plt.subplots(figsize=(COL, 2.1)); style(ax)
x = np.arange(len(BUCKETS))
ax.plot(x, mp.values, color=CAT[0], marker="o", ms=2.6, label="model's own prior")
ax.plot(x, cp.values, color=CAT[1], marker="s", ms=2.6, label="corpus-frequency prior")
ax.axhline(1/9, ls=":", color=MUTED, lw=0.9)
ax.annotate("chance", (0.1, 1/9+0.012), fontsize=6, color=MUTED)
ax.set_xticks(x); ax.set_xticklabels(BUCKETS, rotation=45, ha="right")
ax.set_xlabel("co-occurrences"); ax.set_ylabel("P(wrong answer = prior)")
ax.set_ylim(0, 0.65); ax.legend(frameon=False, loc="upper right")
save(fig, "fig6_prior_collapse", FIGDIR)
T["prior"] = pd.DataFrame({"bucket":BUCKETS,"model_prior":mp.values,"corpus_prior":cp.values})

# ---- Figure 7: calibration (ECE) ----
d = df[(df.model==MAIN)&(df.templates=="main")]
rows=[]
for b in BUCKETS:
    for st in sorted(d.step.unique()):
        s = d[(d.bucket==b)&(d.step==st)]
        rows.append({"bucket":b,"step":st,"ece":ece(s.conf_norm, s.acc_norm)})
E = pd.DataFrame(rows).pivot(index="bucket", columns="step", values="ece").reindex(BUCKETS)
fig, ax = plt.subplots(figsize=(COL, 2.1)); style(ax)
for c, b in zip(ORD5, BSHOW):
    ax.plot(E.columns, E.loc[b].values, color=c, marker="o", ms=2.2, label=b)
ax.set_xscale("symlog", linthresh=1000)
ax.set_xticks([0,1000,10000,100000,658032])
ax.set_xticklabels(["0","1k","10k","100k","658k"])
ax.set_xlabel("training step"); ax.set_ylabel("expected calibration error")
ax.legend(frameon=False, title="co-occurrences", ncol=2, title_fontsize=6.5)
save(fig, "fig7_ece", FIGDIR)
T["ece"] = E


## 10. Statistics

In [ ]:

print("=" * 70); print("STATISTICS"); print("=" * 70)
from scipy import stats as st
d = df[(df.model==MAIN)&(df.templates=="main")&(df.step==FINAL)]
S = {}

rho, p = st.spearmanr(np.log1p(d.n_shared), d.gap_f)
S["spearman_gap_vs_exposure"] = {"rho": float(rho), "p": float(p), "n": int(len(d))}
print(f"fact-level Spearman(log exposure, conf-acc): rho={rho:+.3f}  p={p:.2e}  n={len(d)}")

r0, p0 = st.spearmanr(np.log1p(df[(df.model==MAIN)&(df.templates=='main')&(df.step==0)].n_shared),
                      df[(df.model==MAIN)&(df.templates=='main')&(df.step==0)].gap_f)
S["spearman_at_init"] = {"rho": float(r0), "p": float(p0)}
print(f"same at step 0 (control):                    rho={r0:+.3f}  p={p0:.2e}")

lo = d[d.bucket.isin(["0","1"])].gap_f; hi = d[d.bucket.isin(["61-200","200+"])].gap_f
t, pt = st.mannwhitneyu(lo, hi)
S["rare_vs_frequent"] = {"U": float(t), "p": float(pt),
                         "gap_rare": float(lo.mean()), "gap_frequent": float(hi.mean())}
print(f"rare (0-1) gap {lo.mean():+.3f} vs frequent (61+) {hi.mean():+.3f}: "
      f"Mann-Whitney p={pt:.2e}")

pooled = df[(df.model==MAIN)&(df.templates=='main')].groupby("step")[["acc_norm","conf_norm"]].mean()
post = pooled[pooled.index >= 1000]
S["conf_after_1000"] = {"min": float(post.conf_norm.min()), "max": float(post.conf_norm.max()),
                        "range": float(post.conf_norm.max()-post.conf_norm.min())}
S["acc_after_1000"]  = {"min": float(post.acc_norm.min()),  "max": float(post.acc_norm.max()),
                        "range": float(post.acc_norm.max()-post.acc_norm.min())}
print(f"after step 1000: confidence spans {S['conf_after_1000']['range']:.3f}, "
      f"accuracy spans {S['acc_after_1000']['range']:.3f}")

w = df[(df.model==MAIN)&(df.templates=='main')&(df.acc_norm==0)&(df.step>0)]
S["prior_collapse"] = {"model": float(w.chose_model_prior.mean()),
                       "corpus": float(w.chose_corpus_prior.mean()), "chance": 1/9}
b = st.binomtest(int(w.chose_model_prior.sum()), len(w), 1/9)
S["prior_collapse"]["p"] = float(b.pvalue)
print(f"prior collapse: model {w.chose_model_prior.mean():.3f} vs chance 0.111, p={b.pvalue:.2e}")


## 11. Export

In [ ]:

os.makedirs(TABDIR, exist_ok=True)
for k, v in T.items():
    v.to_csv(os.path.join(TABDIR, f"table_{k}.csv"))
json.dump({"quality": Q, "stats": S}, open(os.path.join(TABDIR, "stats.json"), "w"),
          indent=2, default=float)

print("=" * 70); print("EXPORTED"); print("=" * 70)
for f in sorted(os.listdir(FIGDIR)):
    if f.endswith(".pdf"): print("  fig:", f)
for f in sorted(os.listdir(TABDIR)): print("  tab:", f)
print("\nSend Claude:  the whole", FIGDIR, "and", TABDIR, "folders.")
